# Final Workflow: Roll Rate Model

Notebook gọn nhẹ chỉ giữ code chính:
1. Load data
2. Build transition matrices
3. Build lifecycle + calibration
4. **Allocate TỐI ƯU** xuống loan-level (actual từ df_raw, forecast khi cần)
5. Export reports

**Tối ưu allocation:**
- Cohort có actual @ target_mob: Lấy thực tế từ df_raw ✅
- Cohort chỉ có forecast: Mới allocate ✅
- Kết quả: Nhanh hơn 60%, chính xác hơn

In [1]:
# Setup
import sys
from pathlib import Path
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from datetime import datetime

from src.config import CFG, BUCKETS_CANON, BUCKETS_30P, BUCKETS_90P
from src.config import parse_date_column, create_segment_columns, SEGMENT_COLS
from src.config import K_POST_MATURE  # K value cho MOB > TARGET_MOB
from src.data_loader import load_data
from src.rollrate.transition import compute_transition_by_mob
from src.rollrate.lifecycle import (
    get_actual_all_vintages_amount,
    build_full_lifecycle_amount,
    tag_forecast_rows_amount,
    add_del_metrics,
    aggregate_to_product,
    aggregate_products_to_portfolio,
    lifecycle_to_long_df_amount,
    combine_all_lifecycle_amount,
    export_lifecycle_all_products_one_file,
    extend_actual_info_with_portfolio,
)
from src.rollrate.calibration_kmob import (
    fit_k_raw, smooth_k, fit_alpha,
    forecast_all_vintages_partial_step,
)
from src.rollrate.allocation_v2_optimized import allocate_multi_mob_optimized

from src.rollrate.lifecycle_export_enhanced import export_lifecycle_with_config_info

print("✅ Import thành công")

✅ Import thành công


## 1️⃣ LOAD DATA

In [ ]:
# ========== CẤU HÌNH ==========
#DATA_PATH = 'C:/Users/MAFC4709/Python_work/POS_Parquet_YYYYMM'  # 🔥 Thay đổi path 
DATA_PATH = 'C:/Users/MAFC4709/Python_work/ETB_Parquet_New'  # 🔥 Thay đổi path 
MAX_MOB = 24 # Forecast đến 
TARGET_MOBS = [24]  # Allocate tại MOB nào
 
df_raw = load_data(DATA_PATH)
df_raw['DISBURSAL_DATE'] = parse_date_column(df_raw['DISBURSAL_DATE'])
#df_raw = df_raw[df_raw["PRODUCT_TYPE"].isin(["C","S"])]
df_raw = create_segment_columns(df_raw)

print(f"📊 Data: {len(df_raw):,} rows | {df_raw[CFG['loan']].nunique():,} loans")
print(f"   SEGMENT_COLS: {SEGMENT_COLS}")
print(f"   Products: {df_raw['PRODUCT_TYPE'].unique().tolist()}")
print(f"   Risk scores: {df_raw['RISK_SCORE'].nunique()} unique")

📦 Loading Parquet from: C:\Users\MAFC4709\Python_work\RR_model_1911_Projection\ETB_Parquet_YYYYMM
✅ Loaded 6,283,125 rows via pyarrow.dataset from C:\Users\MAFC4709\Python_work\RR_model_1911_Projection\ETB_Parquet_YYYYMM
📊 Data: 6,283,125 rows | 377,885 loans
   SEGMENT_COLS: ['PRODUCT_TYPE', 'RISK_SCORE']
   Products: ['X', 'T']
   Risk scores: 5 unique


In [3]:
# 1) Đếm số giá trị DISBURSAL_DATE khác nhau
n_unique = df_raw["DISBURSAL_DATE"].nunique(dropna=True)

# 2) Lấy danh sách giá trị unique (giống ý bạn)
unique_vals = df_raw["DISBURSAL_DATE"].dropna().unique()

# 3) Count số dòng theo từng DISBURSAL_DATE (group count)
cnt_by_date = df_raw.groupby("DISBURSAL_DATE", dropna=False).size().reset_index(name="row_count")


In [4]:
cnt_by_date

,DISBURSAL_DATE,row_count
0,2023-01-01,135242
1,2023-02-01,124680
2,2023-03-01,429114
3,2023-04-01,405020
4,2023-05-01,337755
5,2023-06-01,329856
6,2023-07-01,435050
7,2023-08-01,345152
8,2023-09-01,315375
9,2023-10-01,318200


In [5]:
df_raw

,CUTOFF_DATE,AGREEMENT_ID,DISBURSAL_DATE,DISBURSAL_AMOUNT,PRINCIPLE_OUTSTANDING,MOB,TERM_CATEGORY,DPD_GROUP,STATE_MODEL,LA_GROUP,RISK_SCORE,TERM_BY_SCHEME,GENDER,PRODUCT_TYPE,SALE_CHANNEL
0,202307,4335697,2023-02-01,0.030000,0.025601,5,MEDIUM-TERM,DPD0,DPD0,40M-,B,M+,F,X,None
1,202307,4638669,2023-05-01,0.092000,0.090613,2,MEDIUM-TERM,DPD0,DPD0,40M+,A,M+,F,X,None
2,202307,4165912,2023-01-01,0.021540,0.011878,6,SHORT-TERM,DPD0,DPD0,25M-,B,S,M,X,None
3,202307,4466761,2023-04-01,0.070000,0.065163,3,MEDIUM-TERM,DPD0,DPD0,40M+,A,M+,M,X,None
4,202307,4295776,2023-02-01,0.021540,0.013546,5,SHORT-TERM,DPD0,DPD0,25M-,B,S,F,X,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6283120,202601,5503225,2024-05-01,0.010770,0.010770,20,NaN,NaN,DPD90+,12M-,B,M+,F,T,None
6283121,202601,5503239,2024-05-01,0.010770,0.000000,20,NaN,NaN,PREPAY,12M-,D,M+,M,T,None
6283122,202601,5503240,2024-05-01,0.012816,0.000000,20,NaN,NaN,PREPAY,20M-,D,S,M,X,None
6283123,202601,5503247,2024-05-01,0.034464,0.020464,20,NaN,NaN,DPD90+,24M+,A,M+,M,T,None


## 2️⃣ BUILD TRANSITION MATRICES

In [6]:
print("🔨 Building transition matrices...")
matrices_by_mob, parent_fallback = compute_transition_by_mob(df_raw)
print(f"✅ {len(matrices_by_mob)} products | {sum(len(m) for m in matrices_by_mob.values())} matrices")

🔨 Building transition matrices...
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=T, score=A)
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=T, score=B)
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=T, score=C)
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=T, score=D)
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=T, score=NA)
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=X, score=A)
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=X, score=B)
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=X, score=C)
⚠️ Có 2 hàng có tổng weight = 0: ['PREPAY', 'SOLDOUT']
⚙️ Built parent fallback for (product=X, score=D)
⚠️ Có 2 hàng có tổng

## 3️⃣ BUILD LIFECYCLE + CALIBRATION

In [7]:
# ============================
# 3️⃣ BUILD LIFECYCLE + CALIBRATION
# ============================

print("🔨 Calibrating k and alpha...")

# Actual results
actual_results = get_actual_all_vintages_amount(df_raw)

# DISB_TOTAL map
loan_disb = df_raw.groupby(["PRODUCT_TYPE", "RISK_SCORE", CFG["orig_date"], CFG["loan"]])[CFG["disb"]].first()
disb_total_by_vintage = loan_disb.groupby(level=[0, 1, 2]).sum().to_dict()

# Fit k_raw with WLS Regularization (conservative approach)
LAMBDA_K = 1e-4  # Regularization strength
K_PRIOR = 0.0    # Prior value (bias toward 0 for conservative forecast)

k_raw_by_mob, weight_by_mob, _ = fit_k_raw(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    states=BUCKETS_CANON,
    s30_states=BUCKETS_30P,
    include_co=True,
    denom_mode="disb",
    disb_total_by_vintage=disb_total_by_vintage,
    weight_mode="equal",       # Equal weight for all vintages
    method="wls_reg",          # Regularized WLS for stability
    lambda_k=LAMBDA_K,         # Regularization parameter
    k_prior=K_PRIOR,           # Prior value
    min_obs=5,
    fallback_k=1.0,
    fallback_weight=0.0,
    return_detail=True,
)

print(f"   K values: {len(k_raw_by_mob)} MOBs")

# Smooth k
mob_min = min(k_raw_by_mob.keys()) if k_raw_by_mob else 0
mob_max = max(k_raw_by_mob.keys()) if k_raw_by_mob else 0
k_smooth_by_mob, _, _ = smooth_k(k_raw_by_mob, weight_by_mob, mob_min, mob_max)

# Fit alpha
alpha, k_final_by_mob, _ = fit_alpha(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    states=BUCKETS_CANON,
    s30_states=BUCKETS_30P,
    k_smooth_by_mob=k_smooth_by_mob,
    mob_target=min(MAX_MOB, mob_max) if mob_max else MAX_MOB,
    include_co=True,
)

print(f"   Alpha: {alpha:.4f}")
print(f"   K_final: {len(k_final_by_mob)} MOBs")

# ============================
# APPLY K_POST_MATURE (nếu được cấu hình)
# ============================
# K_POST_MATURE: Giá trị K cố định cho MOB > TARGET_MOB
# Mục đích: Giảm slope của DEL curve sau khi mature
# Vì K tăng dần đến 1.0 sẽ làm slope tăng dù P_m đã ổn định

if K_POST_MATURE is not None:
    target_mob = TARGET_MOBS[0] if isinstance(TARGET_MOBS, list) else TARGET_MOBS
    print(f"\n🔧 Applying K_POST_MATURE = {K_POST_MATURE} for MOB >= {target_mob}")
    
    # Lưu K trước khi thay đổi để so sánh
    k_before = {mob: k_final_by_mob.get(mob, 1.0) for mob in range(target_mob, MAX_MOB + 1)}
    
    # Apply K_POST_MATURE cho MOB >= TARGET_MOB
    for mob in range(target_mob, MAX_MOB + 1):
        k_final_by_mob[mob] = K_POST_MATURE
    
    # In so sánh
    print(f"   K values comparison (MOB {target_mob} → {MAX_MOB}):")
    print(f"   {'MOB':<6} {'Before':<10} {'After':<10}")
    print(f"   {'-'*26}")
    for mob in range(target_mob, min(target_mob + 5, MAX_MOB + 1)):
        k_old = k_before.get(mob, 1.0)
        k_new = k_final_by_mob.get(mob, 1.0)
        marker = '← TARGET_MOB' if mob == target_mob else ''
        print(f"   {mob:<6} {k_old:<10.4f} {k_new:<10.4f} {marker}")
    if MAX_MOB > target_mob + 5:
        print(f"   ...")
        print(f"   {MAX_MOB:<6} {k_before.get(MAX_MOB, 1.0):<10.4f} {k_final_by_mob.get(MAX_MOB, 1.0):<10.4f}")
    
    print(f"\n   ✅ K_POST_MATURE applied: MOB {target_mob} → {MAX_MOB} = {K_POST_MATURE}")
else:
    print(f"\n   ℹ️  K_POST_MATURE = None, using calibrated K values")


🔨 Calibrating k and alpha...


C:\Users\MAFC4709\Python_work\RR_model_1911_Projection\src\rollrate\lifecycle.py:858: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_m.groupby(state_col)[ead_col].sum()
C:\Users\MAFC4709\Python_work\RR_model_1911_Projection\src\rollrate\lifecycle.py:858: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_m.groupby(state_col)[ead_col].sum()
C:\Users\MAFC4709\Python_work\RR_model_1911_Projection\src\rollrate\lifecycle.py:858: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to ado

   K values: 24 MOBs
   Alpha: 1.0000
   K_final: 24 MOBs

🔧 Applying K_POST_MATURE = 0.03 for MOB >= 24
   K values comparison (MOB 24 → 24):
   MOB    Before     After     
   --------------------------
   24     1.0000     0.0300     ← TARGET_MOB

   ✅ K_POST_MATURE applied: MOB 24 → 24 = 0.03


In [8]:
# Forecast với k_final
forecast_calibrated = forecast_all_vintages_partial_step(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    max_mob=MAX_MOB,
    k_by_mob=k_final_by_mob,
    states=BUCKETS_CANON,
)

# Combine actual + forecast
lifecycle_combined = combine_all_lifecycle_amount(actual_results, forecast_calibrated)
df_lifecycle_final = lifecycle_to_long_df_amount(lifecycle_combined)
df_lifecycle_final = tag_forecast_rows_amount(df_lifecycle_final, df_raw)
df_lifecycle_final = add_del_metrics(df_lifecycle_final, df_raw)

print(f"✅ Lifecycle: {len(df_lifecycle_final):,} rows | Forecast: {(df_lifecycle_final['IS_FORECAST']==1).sum():,}")

✅ Lifecycle: 8,124 rows | Forecast: 2,732


In [9]:
# ============================
# 4️⃣ AGGREGATE TO PRODUCT & PORTFOLIO
# ============================

# Aggregate to product level
df_product = aggregate_to_product(df_lifecycle_final)

# Aggregate to portfolio level
df_portfolio = aggregate_products_to_portfolio(
    df_product,
    portfolio_name="PORTFOLIO_ALL"
)

# Combine product + portfolio
df_del_all = pd.concat([df_product, df_portfolio], ignore_index=True)

print(f"\n✅ Aggregation complete:")
print(f"   Product-level: {len(df_product):,} rows")
print(f"   Portfolio-level: {len(df_portfolio):,} rows")
print(f"   Combined: {len(df_del_all):,} rows")

# Create actual_info for all products (max actual MOB across scores)
actual_info_prod = (
    df_lifecycle_final[df_lifecycle_final["IS_FORECAST"] == 0]
    .groupby(["PRODUCT_TYPE", "VINTAGE_DATE"])["MOB"]
    .max()
    .to_dict()
)

# Extend with portfolio
actual_info_all = extend_actual_info_with_portfolio(
    actual_info_prod,
    portfolio_name="PORTFOLIO_ALL"
)

print(f"\n✅ Actual info: {len(actual_info_all):,} cohorts")



✅ Aggregation complete:
   Product-level: 1,809 rows
   Portfolio-level: 905 rows
   Combined: 2,714 rows

✅ Actual info: 111 cohorts


## 5️⃣ ALLOCATE TO LOAN-LEVEL (FAST)

In [10]:
print(f"🔨 Allocating to loan-level (MOB {TARGET_MOBS})...")
print("   📌 Sử dụng allocation TỐI ƯU: actual từ df_raw, forecast khi cần")

# Lấy snapshot mới nhất
latest_cutoff = df_raw['CUTOFF_DATE'].max()
df_loans_latest = df_raw[df_raw['CUTOFF_DATE'] == latest_cutoff].copy()
df_loans_latest['VINTAGE_DATE'] = parse_date_column(df_loans_latest[CFG['orig_date']])


# Allocate TỐI ƯU (actual từ df_raw, forecast khi cần)
df_loan_forecast = allocate_multi_mob_optimized(
    df_raw=df_raw,  # ← Thêm df_raw để lấy actual
    df_loans_latest=df_loans_latest,
    df_lifecycle_final=df_lifecycle_final,
    matrices_by_mob=matrices_by_mob,
    target_mobs=TARGET_MOBS,
    parent_fallback=parent_fallback,
    include_del30=False,
    include_del90=True,
    seed=42
)

print(f"\n✅ Allocation: {len(df_loan_forecast):,} loans")
for mob in TARGET_MOBS:
    del90 = df_loan_forecast[f'DEL90_FLAG_MOB{mob}'].mean() * 100
    print(f"   MOB {mob}: DEL90 = {del90:.2f}%")


🔨 Allocating to loan-level (MOB [24])...
   📌 Sử dụng allocation TỐI ƯU: actual từ df_raw, forecast khi cần
🎯 Phân bổ forecast TỐI ƯU tại 1 MOB: [24]
   ✅ Lấy actual từ df_raw khi có
   ✅ Allocate forecast khi cần

📍 Processing MOB 24
   📊 Extracting actual loans @ MOB 24 from df_raw...
      Found 120 cohorts with actual data


KeyboardInterrupt: 

## 6️⃣ EXPORT REPORTS

In [ ]:
print("💾 Exporting...")

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ============================
# 1. Lifecycle với Config Info
# ============================

# Chuẩn bị config params
config_params = {
    "DATA_PATH": DATA_PATH,
    "MAX_MOB": MAX_MOB,
    "TARGET_MOBS": TARGET_MOBS,
    "SEGMENT_COLS": SEGMENT_COLS,
    "MIN_OBS": CFG.get("MIN_OBS", 100),
    "MIN_EAD": CFG.get("MIN_EAD", 100),
    "WEIGHT_METHOD": CFG.get("WEIGHT_METHOD", "exp"),
    "ROLL_WINDOW": CFG.get("ROLL_WINDOW", 20),
    "DECAY_LAMBDA": CFG.get("DECAY_LAMBDA", 0.97),
}

lifecycle_file = output_dir / f"Lifecycle_All_Products_{timestamp}.xlsx"
export_lifecycle_with_config_info(
    df_del_all, 
    actual_info_all, 
    df_raw,
    config_params,
    str(lifecycle_file)
)
print(f"   ✅ {lifecycle_file}")

# ============================
# ============================
# 2. Loan forecast (tự động chia sheet nếu > 1M rows)
# ============================
from src.config import export_loan_forecast_excel

loan_file = output_dir / f"Loan_Forecast_{timestamp}.xlsx"
export_loan_forecast_excel(
    df_loan_forecast, 
    loan_file, 
    target_mobs=TARGET_MOBS,
    include_del_sheets=True
)
print(f"   ✅ {loan_file}")

print("\n🎉 DONE!")
